# Molecular Docking Lab — AutoDock Vina with Biotite

In this lab you will run **molecular docking**: predicting *how* and *how strongly*
a small molecule (the **ligand**) binds to a protein (the **receptor**).

We use **[AutoDock Vina](https://vina.scripps.edu/)** driven from Python through
**[Biotite](https://www.biotite-python.org/)**, prepare structures with **RDKit**
and **PDBFixer**, and visualise everything in 3D with **py3Dmol**.

## What you will learn

1. **System preparation** — fetch a structure, split receptor & ligand
2. **Adding hydrogens** — protonate the receptor *and* the ligand (and *why* it matters)
3. **The search box** — center & size of the region Vina explores
4. **Targeted docking** — dock into a known pocket and score the poses
5. **Validation** — RMSD against the experimental pose, energy/RMSD correlation
6. **3D & 2D interaction analysis** — visualise the pose, list contacting residues & H-bonds
7. **PLIP profiling** — classify every non-covalent interaction of a docked pose
8. **Blind docking** — dock when the pocket is *unknown* (and *see* where it lands)
9. **Virtual screening** — rank a small library of compounds

## Worked example — suvorexant / orexin receptor (PDB `4S0V`)

`4S0V` is the **orexin receptor** (a membrane GPCR) in complex with the
insomnia drug **suvorexant** (ligand code `SUV`). Unlike a small, tight binder,
suvorexant is a **large, flexible drug** sitting in a **membrane** pocket — a
*realistic and genuinely hard* docking target. Because we know the experimental
pose, we can honestly measure how well docking does (spoiler: this one is tough!).

> Note: this crystal structure has **no hydrogen atoms** (typical for X-ray
> structures), so we will have to **add them ourselves** — exactly the step the
> textbook examples often skip.

---

## Exercises overview

| Exercise | Task |
|----------|------|
| 1 | Separate receptor and reference ligand, find the pocket center |
| 2 | Prepare a ligand from SMILES (parse → **add H** → embed → convert) |
| 3 | Run targeted docking with `VinaApp` |
| 4 | Compute pose RMSD and the energy/RMSD correlation |
| 5 | Run **blind** docking and check whether it finds the real pocket |
| 6 | Run a small **virtual screening** and rank the hits |

### References
- Biotite docking example: https://www.biotite-python.org/latest/examples/gallery/structure/modeling/docking.html
- `VinaApp` API: https://www.biotite-python.org/latest/apidoc/biotite.application.autodock.VinaApp.html
- Jupyter Dock (Ruiz-Moreno): https://github.com/AngelRuizMoreno/Jupyter_Dock
- 2D interaction maps: https://chem-workflows.com/content/MolecularDocking.html


## 0. Setup — get a docking engine (Windows · Linux · macOS)

`VinaApp` wraps the **`vina` command-line program**, so the binary must be
available. You have two options.

**Option A — install once from the repo (recommended):**

```bash
uv run scripts/install_docking_tools.py            # downloads Vina into ./bin
uv run scripts/install_docking_tools.py --smina    # also Smina (Linux/macOS)
```

**Option B — let the cell below do it.** It looks for `vina` on your `PATH`,
then in `./bin` (created by the script above), and otherwise downloads the right
prebuilt binary for your OS and CPU (Windows `.exe`, Linux/macOS, x86-64 *or*
arm64/aarch64).

> Native binaries are used on every platform — including Apple-Silicon Macs and
> ARM Linux. If the automatic download is blocked on your network, install Vina
> manually (`conda install -c conda-forge vina`) and set `VINA_BIN` yourself.


In [ ]:
# === SETUP: locate or download a docking engine (Windows / Linux / macOS) ===
import platform, stat, shutil, urllib.request
from pathlib import Path

VINA_VERSION = "1.2.5"


def _vina_asset():
    """Return (release_asset_name, local_filename) for this OS/CPU."""
    system = platform.system()
    arch = "aarch64" if platform.machine().lower() in ("arm64", "aarch64") else "x86_64"
    if system == "Windows":
        return f"vina_{VINA_VERSION}_win.exe", "vina.exe"
    if system == "Darwin":
        return f"vina_{VINA_VERSION}_mac_{arch}", "vina"
    if system == "Linux":
        return f"vina_{VINA_VERSION}_linux_{arch}", "vina"
    raise RuntimeError(f"Unsupported OS {system!r}: install Vina manually, set VINA_BIN")


def ensure_vina():
    exe = "vina.exe" if platform.system() == "Windows" else "vina"
    # 1) Already on PATH?
    found = shutil.which("vina")
    if found:
        return found
    # 2) Installed by scripts/install_docking_tools.py (./bin) or cached nearby?
    for d in (Path("bin"), Path("../bin"), Path(".")):
        p = (d / exe).resolve()
        if p.exists():
            return str(p)
    # 3) Download the right prebuilt binary for this platform
    asset, local_name = _vina_asset()
    url = (f"https://github.com/ccsb-scripps/AutoDock-Vina/releases/"
           f"download/v{VINA_VERSION}/{asset}")
    dest = Path(local_name).resolve()
    print(f"Downloading AutoDock Vina:\n  {url}")
    urllib.request.urlretrieve(url, dest)
    if platform.system() != "Windows":
        dest.chmod(dest.stat().st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    return str(dest)


VINA_BIN = ensure_vina()
print("Using vina binary:", VINA_BIN)


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import biotite.database.rcsb as rcsb
import biotite.structure as struc
import biotite.structure.info as info
import biotite.structure.io.pdb as pdb
import biotite.structure.io.pdbx as pdbx
import biotite.application.autodock as autodock
import biotite.interface.rdkit as rdkit_interface

from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import py3Dmol

# Configuration for the worked example
PDB_ID = "4S0V"          # orexin receptor + suvorexant
LIGAND_RESNAME = "SUV"   # suvorexant
CHAIN = "A"              # 4S0V is a single monomer (chain A)
RANDOM_SEED = 0
BOX_SIZE = [20.0, 20.0, 20.0]   # Å, edge lengths of the search box
print("Imports OK")


## 2. Fetch and inspect the complex

We download `4S0V` directly from the RCSB PDB. Important details:

- **`include_bonds=True`** — `VinaApp` requires a `BondList` for both molecules.
- **`extra_fields=["charge"]`** — formal charges improve partial-charge estimation.
- `4S0V` is a single monomer (chain `A`); besides the protein and the `SUV`
  ligand it also contains water (`HOH`) and an oleic-acid lipid (`OLA`) — a hint
  that this is a **membrane** protein.


In [ ]:
pdbx_file = pdbx.BinaryCIFFile.read(rcsb.fetch(PDB_ID, "bcif"))
structure = pdbx.get_structure(
    pdbx_file,
    model=1,
    include_bonds=True,        # VinaApp needs a BondList
    extra_fields=["charge"],   # formal charges -> better partial charges
)
structure = structure[structure.chain_id == CHAIN]

print(f"{PDB_ID} chain {CHAIN}: {structure.array_length()} atoms")
print("Hetero residues:", np.unique(structure.res_name[structure.hetero]))
print("Ligand full name:", info.full_name(LIGAND_RESNAME))


## 3. Exercise 1 — Separate receptor and reference ligand

Split the complex into:

- **`receptor_raw`** — all amino-acid atoms (hint: `struc.filter_amino_acids`).
  We call it *raw* because it still lacks hydrogens (we fix that next).
- **`ref_ligand`** — suvorexant, `res_name == LIGAND_RESNAME` (the *experimental* pose)
- **`pocket_center`** — the centroid of the reference ligand (hint: `struc.centroid`)

The reference pose lets us *measure* docking accuracy later.


In [ ]:
# TODO: receptor_raw = amino-acid atoms only
receptor_raw = ...    # TODO

# TODO: ref_ligand = atoms whose res_name == LIGAND_RESNAME
ref_ligand = ...      # TODO

# TODO: pocket_center = centroid of the reference ligand
pocket_center = ...   # TODO

# print(f"Receptor (no H) atoms:  {receptor_raw.array_length()}")
# print(f"Reference ligand atoms: {ref_ligand.array_length()}")
# print(f"Pocket center (xyz):    {np.round(pocket_center, 2)}")


## 4. Receptor preparation — **add hydrogens**

Two things docking *silently* depends on:

1. **Hydrogen atoms.** Vina keeps **polar** hydrogens (H-bond donors) and the
   partial charges are computed from the protonated structure.
2. **Complete residues.** X-ray structures often miss side-chain atoms.

`4S0V` has **zero** hydrogens, so we must protonate it. We use **PDBFixer**
(part of the OpenMM ecosystem) to rebuild missing heavy atoms and add hydrogens
at pH 7. This is the step the minimal biotite example skips.

The helper below is provided (the PDBFixer API is not the point of the lab).


In [ ]:
def protonate_receptor(receptor_heavy, ph=7.0):
    """Add missing heavy atoms + hydrogens to a protein AtomArray (via PDBFixer)."""
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile

    # biotite AtomArray -> PDB -> PDBFixer
    pf = pdb.PDBFile()
    pf.set_structure(receptor_heavy)
    buf = StringIO()
    pf.write(buf)
    buf.seek(0)

    fixer = PDBFixer(pdbfile=buf)
    fixer.findMissingResidues()
    fixer.missingResidues = {}                 # don't model whole missing loops
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()                     # rebuild missing side-chain atoms
    fixer.addMissingHydrogens(ph)              # <-- add hydrogens

    # PDBFixer -> PDB -> biotite AtomArray (+ bonds for VinaApp)
    out = StringIO()
    PDBFile.writeFile(fixer.topology, fixer.positions, out)
    out.seek(0)
    protonated = pdb.PDBFile.read(out).get_structure(model=1)
    protonated.bonds = struc.connect_via_residue_names(protonated)
    return protonated


receptor = protonate_receptor(receptor_raw, ph=7.0)
n_h = int(np.count_nonzero(receptor.element == "H"))
print(f"Before: {receptor_raw.array_length()} atoms, 0 H")
print(f"After : {receptor.array_length()} atoms, {n_h} H  <- ready to dock")


## 5. Ligand preparation

A docking ligand needs **3D coordinates, hydrogens, bonds and charges**. There are
two common starting points.

### 5a. From the Chemical Component Dictionary (CCD)
For ligands already in the PDB, `info.residue()` returns a clean, protonated
template with bonds and charges. Its atom names match `ref_ligand` (`SUV`), so we
can compare poses directly — this is what we dock in the worked example.


In [ ]:
ligand_ccd = info.residue(LIGAND_RESNAME)
print("CCD suvorexant atoms:", ligand_ccd.array_length(),
      "| hydrogens:", int(np.count_nonzero(ligand_ccd.element == "H")))


### 5b. Exercise 2 — From a SMILES string (the general case)

Real projects rarely have a ready 3D ligand. The standard preparation pipeline is:

1. Parse the SMILES — `Chem.MolFromSmiles`
2. **Add hydrogens** — `Chem.AddHs`  ← *crucial and easy to forget!*
3. Generate a 3D conformer — `AllChem.EmbedMolecule(mol, randomSeed=seed)`
4. Optimise the geometry — `AllChem.MMFFOptimizeMolecule(mol)`
5. Convert to a Biotite `AtomArray` — `rdkit_interface.from_mol(mol)[0]`
   (`from_mol` returns an `AtomArrayStack`; take model `0`)

Complete `smiles_to_ligand` below.


In [ ]:
def smiles_to_ligand(smiles, seed=RANDOM_SEED):
    """SMILES -> protonated, 3D, biotite AtomArray (with bonds + charges)."""
    mol = Chem.MolFromSmiles(smiles)
    # TODO: 2) add hydrogens
    # TODO: 3) embed a 3D conformer (use randomSeed=seed)
    # TODO: 4) optimise the geometry (MMFF)
    # TODO: 5) convert to a biotite AtomArray (model 0 of the stack)
    ligand = ...   # TODO
    return ligand


# Suvorexant (ligand SUV)
SUV_SMILES = "Cc1ccc(-n2nccn2)c(C(=O)N2CCN(c3nc4cc(Cl)ccc4o3)CCC2C)c1"
# ligand = smiles_to_ligand(SUV_SMILES)
# print("Prepared ligand atoms:", ligand.array_length(),
#       "| hydrogens:", int(np.count_nonzero(ligand.element == "H")))


## 6. The search box

Vina only explores a **box** in space, defined by a **center** and a **size**
(edge lengths in Å). For *targeted* docking we center the box on the known
pocket. The box must be large enough to contain the (fairly large) suvorexant
molecule in any orientation, but a smaller box makes the search faster and more
focused.

The helper below renders the receptor, the reference ligand, and the box.


In [ ]:
def to_pdb_str(atoms):
    """Serialise an AtomArray to a PDB string for py3Dmol."""
    f = pdb.PDBFile()
    f.set_structure(atoms)
    s = StringIO()
    f.write(s)
    return s.getvalue()


def add_box(view, center, size, color="magenta"):
    cx, cy, cz = (float(v) for v in center)
    sx, sy, sz = (float(v) for v in size)
    view.addBox({
        "center": {"x": cx, "y": cy, "z": cz},
        "dimensions": {"w": sx, "h": sy, "d": sz},
        "color": color, "opacity": 0.5, "wireframe": True,
    })


view = py3Dmol.view(width=720, height=520)
view.addModel(to_pdb_str(receptor), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
view.addModel(to_pdb_str(ref_ligand), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
add_box(view, pocket_center, BOX_SIZE)
view.zoomTo({"model": 1})
view.show()


## 7. Exercise 3 — Run targeted docking

Dock the CCD suvorexant (`ligand_ccd`) into the pocket with `VinaApp`:

```python
app = autodock.VinaApp(ligand, receptor, center, size, bin_path=VINA_BIN)
```

Then configure and run it:

- `app.set_seed(RANDOM_SEED)` — reproducible results
- `app.set_exhaustiveness(16)` — search effort (higher = better but slower;
  suvorexant is flexible, so we use more effort than the default 8)
- `app.set_max_number_of_models(12)` — how many poses to return
- `app.set_energy_range(6.0)` — keep poses within 6 kcal/mol of the best
- `app.start()` then `app.join()`
- read `app.get_ligand_coord()` and `app.get_energies()`


In [ ]:
# app = autodock.VinaApp(...)        # TODO: build the app (use VINA_BIN)
# TODO: set_seed / set_exhaustiveness / set_max_number_of_models / set_energy_range
# TODO: start() and join()

# docked_coord = ...   # TODO: app.get_ligand_coord()
# energies     = ...   # TODO: app.get_energies()

# print("Generated", len(energies), "binding modes")
# print("Best predicted affinity:", round(float(energies.min()), 2), "kcal/mol")


Vina discards nonpolar hydrogens, so their coordinates come back as `NaN`.
We assemble an `AtomArrayStack` of all poses and drop those atoms.

In [ ]:
docked_ligand = struc.from_template(ligand_ccd, docked_coord)
# Remove nonpolar-H atoms whose coordinates are NaN
docked_ligand = docked_ligand[..., ~np.isnan(docked_ligand.coord[0]).any(axis=-1)]

modes = pd.DataFrame({
    "mode": np.arange(1, len(energies) + 1),
    "affinity_kcal_per_mol": np.round(energies, 2),
})
print("Docked stack shape (models, atoms):", docked_ligand.shape)
modes


## 8. Exercise 4 — Validate against the experimental pose

Because we know the true pose (`ref_ligand`), we can compute the **RMSD** of each
docked mode to it. A docked pose is usually considered *correct* when RMSD < 2 Å.

> ⚠️ **Reality check:** suvorexant is a large, flexible drug in a membrane
> pocket. Don't be surprised if even the best pose is **several Å** from the
> crystal structure — reproducing such poses is genuinely hard for rigid-receptor
> docking. That difficulty is exactly the lesson here.

First we match the two atom sets and standardise atom order (provided — fiddly but
not the point of the exercise). Then:

- compute `rmsd` of every mode vs the reference with `struc.rmsd`
- compute the Spearman correlation between `energies` and `rmsd`


In [ ]:
# --- provided: align atom sets so RMSD compares like-for-like ---
dl = docked_ligand[..., np.isin(docked_ligand.atom_name, ref_ligand.atom_name)]
dl = dl[..., info.standardize_order(dl)]
rl = ref_ligand[np.isin(ref_ligand.atom_name, dl.atom_name)]
rl = rl[info.standardize_order(rl)]
print("Atoms compared:", dl.array_length())


In [ ]:
# TODO: RMSD of every docked mode vs the reference  (struc.rmsd(rl, dl))
rmsd = ...          # TODO

# TODO: Spearman correlation between energies and rmsd
corr, p = ...       # TODO  (spearmanr returns (statistic, pvalue))

# print("Best RMSD over all modes:", round(float(rmsd.min()), 2), "A")
# print(f"Spearman(energy, RMSD) = {corr:.2f} (p = {p:.3f})")


In [ ]:
# Plot energy vs RMSD (uncomment once `rmsd`/`corr` exist)
# fig, ax = plt.subplots(figsize=(7, 5))
# ax.scatter(energies, rmsd, c="black", marker="+", s=80)
# ax.axhline(2.0, ls="--", color="red", lw=1, label="2 A success threshold")
# ax.set_xlabel("Predicted affinity (kcal/mol)")
# ax.set_ylabel("RMSD to reference (A)")
# ax.set_title(f"Spearman r = {corr:.2f} (p = {p:.3f})")
# ax.legend(); plt.tight_layout(); plt.show()


## 9. Visualise the best pose in 3D

We overlay the **experimental** pose (blue) with our **best docked** pose (green)
inside the receptor surface. Spin it around: is the docked pose in the right
pocket? Is it just rotated/flipped relative to the crystal pose (which inflates
RMSD even when the location is right)?


In [ ]:
best_pose = docked_ligand[0]

view = py3Dmol.view(width=720, height=520)
view.addModel(to_pdb_str(receptor), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "white"}})
view.addSurface(py3Dmol.VDW, {"opacity": 0.55, "color": "white"}, {"model": 0})
# Experimental reference (blue) and docked best pose (green)
view.addModel(to_pdb_str(ref_ligand), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "blueCarbon"}})
view.addModel(to_pdb_str(best_pose), "pdb")
view.setStyle({"model": 2}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 2})
view.show()


## 10. Interaction analysis — 2D table & map

A pose is only useful if it makes sensible interactions. Full interaction
diagrams (LigPlot, ProLIF) need extra packages, so here we build a lightweight
equivalent from Biotite alone:

1. a **contact table** of receptor residues near the ligand (≤ 4 Å),
2. an **H-bond table** between ligand and receptor (this is where adding receptor
   hydrogens pays off!), and
3. a **2D depiction** of the ligand (the "map").


In [ ]:
def interaction_table(receptor, pose, cutoff=4.0):
    """Receptor residues within `cutoff` A of any ligand atom."""
    cell = struc.CellList(receptor, cell_size=cutoff + 1)
    nearest = {}
    for c in pose.coord:
        for j in cell.get_atoms(c, radius=cutoff):
            if j < 0:
                continue
            d = float(np.linalg.norm(c - receptor.coord[j]))
            key = (int(receptor.res_id[j]), str(receptor.res_name[j]))
            if key not in nearest or d < nearest[key]:
                nearest[key] = d
    df = pd.DataFrame(
        [(rid, rname, round(d, 2)) for (rid, rname), d in nearest.items()],
        columns=["res_id", "res_name", "min_dist_A"],
    )
    return df.sort_values("min_dist_A").reset_index(drop=True)


contacts = interaction_table(receptor, best_pose, cutoff=4.0)
print(f"{len(contacts)} residues within 4 A of the docked ligand:")
contacts


In [ ]:
# Hydrogen bonds between the docked ligand and the receptor
complex_ = receptor + best_pose
lig_mask = np.zeros(complex_.array_length(), dtype=bool)
lig_mask[receptor.array_length():] = True

triplets = struc.hbond(complex_, selection1=lig_mask, selection2=~lig_mask)

hb_rows = []
for d_i, h_i, a_i in triplets:
    for idx in (d_i, a_i):                 # report the receptor partner(s)
        if not lig_mask[idx]:
            hb_rows.append((int(complex_.res_id[idx]),
                            str(complex_.res_name[idx]),
                            str(complex_.atom_name[idx])))
hb_df = (pd.DataFrame(sorted(set(hb_rows)),
                      columns=["res_id", "res_name", "atom"])
         if hb_rows else pd.DataFrame(columns=["res_id", "res_name", "atom"]))
print(f"{len(triplets)} ligand-receptor H-bonds involving these atoms:")
hb_df


In [ ]:
# 2D depiction of the ligand (the "interaction map" base layer)
mol2d = Chem.MolFromSmiles(SUV_SMILES)
Draw.MolToImage(mol2d, size=(460, 340))


## 11. Interaction analysis with PLIP

The biotite contact list was a quick approximation. **PLIP** (Protein–Ligand
Interaction Profiler) is a dedicated tool that classifies *all* non-covalent
interactions — hydrophobic contacts, hydrogen bonds, π-stacking, π-cation
interactions, salt bridges, halogen bonds and water bridges — and is a standard
way to report docking results.

PLIP needs **OpenBabel** (provided cross-platform by the `openbabel-wheel`
dependency) plus the PLIP source. The setup cell imports it, cloning the PLIP
repository into `./bin/plip` on first use — this works on **Windows, Linux and
macOS**. You can also pre-fetch it from the repo:

```bash
uv run scripts/install_docking_tools.py --plip
```

> Reference: Schake, Bolz, *et al.*, *Nucleic Acids Research* (2025),
> https://doi.org/10.1093/nar/gkaf361 · https://github.com/pharmai/plip


In [ ]:
# === PLIP setup (cross-platform): OpenBabel wheel + PLIP from source ===
import sys, subprocess
from pathlib import Path


def ensure_plip():
    try:
        import plip  # noqa: F401  (already importable)
        return
    except ImportError:
        pass
    try:
        from openbabel import pybel  # noqa: F401  (from the 'openbabel-wheel' wheel)
    except ImportError as exc:
        raise RuntimeError(
            "OpenBabel is required for PLIP. Run 'uv sync' (it is in pyproject.toml) "
            "or 'uv pip install openbabel-wheel lxml'."
        ) from exc
    # Reuse an existing checkout, otherwise clone PLIP (pure Python -> PYTHONPATH)
    for d in (Path("bin/plip"), Path("../bin/plip")):
        if (d / "plip").is_dir():
            sys.path.insert(0, str(d.resolve()))
            return
    repo_bin = next((c for c in (Path("bin"), Path("../bin")) if c.exists()), Path("bin"))
    repo_bin.mkdir(exist_ok=True)
    target = (repo_bin / "plip").resolve()
    print("Cloning PLIP into", target, "...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/pharmai/plip.git", str(target)],
        check=True,
    )
    sys.path.insert(0, str(target))


ensure_plip()
from plip.structure.preparation import PDBComplex
print("PLIP ready")


PLIP reads a PDB containing both the protein and the ligand (as `HETATM`).
We combine the protonated receptor with our best docked pose, tag the ligand as a
generic `LIG` residue, and let PLIP detect the interactions.

In [ ]:
def run_plip(receptor, pose, lig_resname="LIG", lig_chain="L"):
    """Profile a docked pose with PLIP; return its interaction set."""
    lig = pose.copy()
    lig.hetero[:] = True
    lig.res_name[:] = lig_resname
    lig.chain_id[:] = lig_chain
    lig.res_id[:] = 1

    f = pdb.PDBFile()
    f.set_structure(receptor + lig)
    buf = StringIO()
    f.write(buf)

    complex_ = PDBComplex()
    complex_.load_pdb(buf.getvalue(), as_string=True)
    complex_.analyze()
    return list(complex_.interaction_sets.values())[0]


site = run_plip(receptor, best_pose)
print("PLIP analysed the docked suvorexant pose.")


In [ ]:
def plip_table(site):
    """Flatten PLIP interactions into a tidy DataFrame."""
    groups = [
        ("hydrophobic",  list(site.hydrophobic_contacts), "distance"),
        ("h-bond",       list(site.hbonds_ldon) + list(site.hbonds_pdon), "distance_ad"),
        ("pi-stacking",  list(site.pistacking), "distance"),
        ("pi-cation",    list(site.pication_laro) + list(site.pication_paro), "distance"),
        ("salt-bridge",  list(site.saltbridge_lneg) + list(site.saltbridge_pneg), "distance"),
        ("halogen",      list(site.halogen_bonds), "distance"),
        ("water-bridge", list(site.water_bridges), "distance_aw"),
    ]
    rows = []
    for itype, items, dattr in groups:
        for it in items:
            rows.append({
                "type": itype,
                "residue": f"{it.restype}{it.resnr}{it.reschain}",
                "distance_A": round(float(getattr(it, dattr)), 2),
            })
    return pd.DataFrame(rows, columns=["type", "residue", "distance_A"])


plip_df = plip_table(site)
print("Interactions by type:")
print(plip_df["type"].value_counts().to_string() if len(plip_df) else "  (none detected)")
plip_df


### 3D interaction map
Dashed lines mark the detected interactions (grey = hydrophobic, blue = H-bond,
orange = π-stacking / π-cation, yellow = salt bridge). Interacting residues are
shown as sticks; the docked ligand is green.

In [ ]:
def plip_view(receptor, pose, site, width=760, height=560):
    color = {"hydrophobic": "grey", "hbond": "blue", "pistack": "orange",
             "pication": "orange", "saltbridge": "yellow"}

    view = py3Dmol.view(width=width, height=height)
    view.addModel(to_pdb_str(receptor), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "white"}})

    interacting = (list(site.hydrophobic_contacts)
                   + list(site.hbonds_ldon) + list(site.hbonds_pdon)
                   + list(site.pistacking)
                   + list(site.saltbridge_lneg) + list(site.saltbridge_pneg)
                   + list(site.pication_laro) + list(site.pication_paro)
                   + list(site.halogen_bonds))
    resi = sorted({str(it.resnr) for it in interacting})
    if resi:
        view.addStyle({"model": 0, "resi": resi},
                      {"stick": {"colorscheme": "whiteCarbon"}})

    view.addModel(to_pdb_str(pose), "pdb")
    view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

    def dash(p1, p2, c):
        view.addCylinder({
            "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
            "end":   {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
            "radius": 0.07, "color": c, "dashed": True, "fromCap": 1, "toCap": 1,
        })

    for it in site.hydrophobic_contacts:
        dash(it.ligatom.coords, it.bsatom.coords, color["hydrophobic"])
    for it in list(site.hbonds_ldon) + list(site.hbonds_pdon):
        dash(it.a.coords, it.d.coords, color["hbond"])
    for it in site.pistacking:
        dash(it.ligandring.center, it.proteinring.center, color["pistack"])
    for it in list(site.saltbridge_lneg) + list(site.saltbridge_pneg):
        dash(it.positive.center, it.negative.center, color["saltbridge"])
    for it in list(site.pication_laro) + list(site.pication_paro):
        dash(it.ring.center, it.charge.center, color["pication"])

    view.zoomTo({"model": 1})
    return view


plip_view(receptor, best_pose, site).show()


## 12. Exercise 5 — Blind docking

Often we **don't know** where a ligand binds. *Blind docking* searches the whole
protein by centering a large box on the receptor centroid.

Trade-offs: a bigger box is a much larger search space, so increase
`exhaustiveness` to compensate (slower but more reliable).

Tasks:
- `blind_center = struc.centroid(receptor)`
- use a large `blind_size` (e.g. `[30, 30, 30]`)
- run `VinaApp` with `set_exhaustiveness(12)`
- check whether the best blind pose lands in the *true* pocket by measuring the
  distance between its centroid and `pocket_center`


In [ ]:
# blind_center = ...                 # TODO: centroid of the whole receptor
# blind_size   = [30.0, 30.0, 30.0]
# blind_app = autodock.VinaApp(ligand_ccd, receptor, blind_center, blind_size,
#                              bin_path=VINA_BIN)
# TODO: set_seed, set_exhaustiveness(12), set_max_number_of_models(9), start, join
# blind_coord    = ...   # TODO
# blind_energies = ...   # TODO

# blind_best = struc.from_template(ligand_ccd, blind_coord)[0]
# blind_best = blind_best[~np.isnan(blind_best.coord).any(axis=-1)]
# dist_to_pocket = float(np.linalg.norm(struc.centroid(blind_best) - pocket_center))
# print("Blind best affinity:", round(float(blind_energies.min()), 2), "kcal/mol")
# print("Distance to true pocket:", round(dist_to_pocket, 2), "A")


**Where did the blind pose actually land?** Let's *see* it instead of just
reading a number. Blue = the true crystal pose (real pocket); orange = the best
blind-docking pose; the spheres mark the two pocket centers and the grey wireframe
is the blind search box.

In [ ]:
# Visualise the blind result (uncomment after Exercise 5 is implemented)
# view = py3Dmol.view(width=760, height=560)
# view.addModel(to_pdb_str(receptor), "pdb")
# view.setStyle({"model": 0}, {"cartoon": {"color": "white"}})
# view.addModel(to_pdb_str(ref_ligand), "pdb")           # true pose (blue)
# view.setStyle({"model": 1}, {"stick": {"colorscheme": "blueCarbon"}})
# view.addModel(to_pdb_str(blind_best), "pdb")           # blind pose (orange)
# view.setStyle({"model": 2}, {"stick": {"colorscheme": "orangeCarbon"}})
#
# def _sphere(c, color):
#     view.addSphere({"center": {"x": float(c[0]), "y": float(c[1]), "z": float(c[2])},
#                     "radius": 1.4, "color": color, "opacity": 0.7})
# _sphere(pocket_center, "blue")
# _sphere(struc.centroid(blind_best), "orange")
# add_box(view, blind_center, blind_size, color="grey")
# view.zoomTo()
# view.show()


## 13. Exercise 6 — Virtual screening

Docking shines at **ranking** many candidates. Here we dock a small library into
the suvorexant pocket and rank by best predicted affinity (more negative = better).
Suvorexant is large and high-affinity, so it should clearly out-score the small
drug-like decoys — a quick sanity check of the whole pipeline.

`dock_one` (provided) wires together your `smiles_to_ligand` and `VinaApp`.
Your task: loop over the library, collect the scores, and build a sorted
`DataFrame`.


In [ ]:
def dock_one(smiles, center, size, seed=RANDOM_SEED, exhaustiveness=8):
    """Dock a SMILES into the pocket; return the best predicted affinity."""
    lig = smiles_to_ligand(smiles, seed=seed)
    app = autodock.VinaApp(lig, receptor, center, size, bin_path=VINA_BIN)
    app.set_seed(seed)
    app.set_exhaustiveness(exhaustiveness)
    app.set_max_number_of_models(5)
    app.start()
    app.join()
    return float(app.get_energies().min())


library = {
    "suvorexant": SUV_SMILES,                       # the real ligand (positive control)
    "ibuprofen":  "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
    "aspirin":    "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine":   "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "benzene":    "c1ccccc1",
    "ethanol":    "CCO",
}


In [ ]:
# TODO: dock every compound in `library` into the pocket and collect
#       (name, smiles, best_affinity) rows; then sort ascending by affinity.
# results = []
# for name, smi in library.items():
#     aff = dock_one(smi, pocket_center, BOX_SIZE)
#     results.append((name, smi, round(aff, 2)))
#     print(f"{name:12s} {aff:6.2f} kcal/mol")
# screen = (pd.DataFrame(results, columns=["name", "smiles", "best_affinity"])
#           .sort_values("best_affinity").reset_index(drop=True))
# screen


In [ ]:
# Bar chart of the ranking (uncomment once `screen` exists)
# fig, ax = plt.subplots(figsize=(7, 4))
# ax.barh(screen["name"], screen["best_affinity"], color="steelblue")
# ax.set_xlabel("Best predicted affinity (kcal/mol)")
# ax.set_title("Virtual screening ranking (more negative = better)")
# ax.invert_yaxis(); plt.tight_layout(); plt.show()


## 14. Wrap-up & going further

You built a full docking pipeline: **prepare → add hydrogens → box → dock →
score → validate → visualise → screen** — on a realistically difficult target.

### Discussion
- The best docked pose was several Å from the crystal structure. Which factors
  make suvorexant / `4S0V` hard (ligand flexibility, rigid receptor, membrane
  pocket, scoring-function limits)?
- Why is the energy/RMSD correlation often weak even when the location is right?
- How did adding receptor hydrogens change the H-bond analysis?
- What interaction types did PLIP report, and do they match the known binding mode?
- Blind docking found (or missed) the pocket — what would make it more reliable?

### Going further
- **Flexible side chains:** pass a boolean `flexible` mask to `VinaApp`.
- **Bigger libraries:** wrap `dock_one` in a loop over a CSV of SMILES.
- **Richer 2D maps:** install [ProLIF](https://prolif.readthedocs.io/) for full
  protein-ligand interaction fingerprints and diagrams.
- **Other targets:** swap `PDB_ID` / `LIGAND_RESNAME` for your own system.

### References
- Biotite docking example & `VinaApp` API (see top of notebook)
- Jupyter Dock: https://github.com/AngelRuizMoreno/Jupyter_Dock
- 2D interaction maps: https://chem-workflows.com/content/MolecularDocking.html
